# 01 · Visão geral dos resultados

F1 oficial do LoCoMo por categoria e condição, e as três comparações que
estruturam o estudo:

| comparação | isola |
|---|---|
| B3 → G1 | valor das faces (mesmos fatos extraídos) |
| G1 → G2 | valor da curadoria + consolidação |
| G2 → G3 | valor do sigma-agent |

Pré-requisito: `fgl run-all` (ou `fgl run-all --dry-run` + `setup(dry=True)`).


In [ ]:
from nbutils import *

# results/ por padrão; use setup(dry=True) para inspecionar um smoke run offline
ctx = setup()
ctx.conditions


## Tabela principal


In [ ]:
ctx.f1.pivot_table(index='condition', columns='category', values='f1', observed=True).round(3)


In [ ]:
ctx.overall.round(3)


## F1 por categoria


In [ ]:
plot_f1_by_category(ctx); show()


## O que cada ingrediente adiciona

Barras acima de zero significam que o ingrediente ajudou naquela categoria.


In [ ]:
plot_deltas(ctx); show()


## Taxa de abstenção

Em `adversarial` abster-se **é** a resposta certa. Nas demais categorias,
abstenção alta indica que a recuperação não está encontrando a evidência.


In [ ]:
piv = ctx.f1.pivot_table(index='category', columns='condition',
                        values='abstention_rate', observed=True)
ax = piv.plot(kind='bar', figsize=(10, 3.6), color=colors(piv.columns), width=.82)
ax.set_ylabel('taxa de abstenção'); ax.set_xlabel('')
ax.set_title('Com que frequência o modelo responde "Not mentioned"')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.setp(ax.get_xticklabels(), rotation=15, ha='right'); show()


## Onde as condições discordam

As perguntas em que uma condição acerta e a outra erra — é aqui que se lê
*qualitativamente* o que a topologia está fazendo.


In [ ]:
preds = ctx.all_predictions()
if preds.empty:
    print('sem predictions.jsonl')
else:
    a, b = 'B3-rag-facts', 'G1-fatgraph-min'
    have = set(preds.condition.unique())
    if {a, b} <= have:
        key = ['question', 'category_name', 'gold']
        A = preds[preds.condition == a].set_index(key)
        B = preds[preds.condition == b].set_index(key)
        join = A[['prediction', 'f1']].join(
            B[['prediction', 'f1']], lsuffix=f'_{a}', rsuffix=f'_{b}', how='inner')
        join['delta'] = join[f'f1_{b}'] - join[f'f1_{a}']
        display(join.sort_values('delta', ascending=False).head(10))
        display(join.sort_values('delta').head(10))
    else:
        print(f'preciso de {a} e {b}; tenho {sorted(have)}')


## Relatório completo em Markdown

Idêntico ao que `fgl report` imprime — mesma função, mesmos números.


In [ ]:
print(ctx.report())
